<a href="https://colab.research.google.com/github/Damini-Walia/fcc_predict_health_costs_with_regression/blob/main/fcc_predict_health_costs_with_regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Predit health costs with regression

> In this challenge, you will predict healthcare costs using a regression algorithm.
>
> You are given a dataset that contains information about different people including their healthcare costs. Use the data to predict healthcare costs based on new data.

By Damini Walia

In [ ]:
# Check if seaborn is installed
!pip list | grep seaborn

In [ ]:
# Import libraries. You may or may not use all of these.
# !pip install -q git+https://github.com/tensorflow/docs
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

'''
try:
  # %tensorflow_version only exists in Colab
  %tensorflow_version 2.x
except Exception
  pass
'''

import tensorflow as tf
from tensorflow import keras
from keras import layers

# import tensorflow_docs as tfdocs
# import tensorflow_docs.plots
# import tensorflow_docs.modeling

# function that helps us split up our dataset into a training
# and a validation/testing part
from sklearn.model_selection import train_test_split

In [ ]:
# Import data
!wget https://cdn.freecodecamp.org/project-data/health-costs/insurance.csv
ds = dataset = pd.read_csv('insurance.csv')
dataset.tail()

In [ ]:
# Check if there are any rows containing any NaN values
ds[ds.isna().any(axis=1)]

The dataset doesn't seem to have any missing entries.

Next check what columns containg categorical data (i.e. the datatype is object).
According to the instructions those columns should be converted to numbers

In [ ]:
ds.dtypes

The object fields are
1. sex
2. smoker
3. region

## Smokers

In [ ]:
ds.smoker

Smokers are categorized as yes or no, value should be converted to 1's and 0's (i.e. one-hot encoding)

In [ ]:
ds.smoker = ds.smoker.replace({'yes': 1, 'no': 0})
ds

In [ ]:
ds.dtypes

The datatype for smoker was automatically converted to `int64`

## sex

In [ ]:
ds.sex

In [ ]:
# I'm going to go against the Freudian inclination ;)
ds.sex = ds.sex.replace({'female': 1, 'male': 0})
ds

## Region

1. Let's see what regions there are, and encode them in some schema.

In [ ]:
ds['region'].unique()

Apparently there are 4 regions
1. southwest
2. southeast
3. northwest
4. northeast

In [ ]:
# Don't do this - do one-hot encoding instead.
# ds.region = ds.region.replace(
#     {'southwest': 1, 'southeast': 2, 'northwest': 3, 'northeast': 4})

ds = pd.get_dummies(ds, columns=['region'], prefix='', prefix_sep='', dtype='uint8')
ds.tail()

In [ ]:
ds.dtypes

## Examining the data

In [ ]:
# Computationally intensive
sns.pairplot(ds[['expenses','age', 'sex', 'smoker', 'bmi','children']], diag_kind='kde')

## Training and evaluation

The dataset should be split into a training and validation dataset (80:20)

In [ ]:
train_ds, val_ds = train_test_split(ds, test_size=0.2, random_state=42)

In [ ]:
train_ds

In [ ]:
val_ds

In [ ]:
# Control
100*(268)/(268+1070)

Separate out the Y-variables from the x-variables and get the X/Y-values as numpy arrays

In [ ]:
'''
# 1. Y-variables
train_Y = train_ds.iloc[:, -1].values
val_Y = val_ds.iloc[:, -1].values
print("First 10 training values for expenses:",train_Y[:10])
print("First 10 validation values for expenses:", val_Y[:10])
print("Python types train/val_Y:", type(train_Y), type(val_Y))
'''

In [ ]:
'''
# 2. X-variables

train_X = train_ds.iloc[:, :-1].values
val_X = val_ds.iloc[:, :-1].values

print("First 10 X training values:\n",train_X[:10])
print("First 10 X validation values:\n", val_X[:10])
'''

In [ ]:
train_dataset = train_ds.copy()
test_dataset = val_ds.copy()

train_labels = train_dataset.pop('expenses')
test_labels = test_dataset.pop('expenses')

In [ ]:
# Series should containg expenses for the training dataset
train_labels

In [ ]:
# Should no longer contain the expenses column
train_dataset

## Feature normalization

It is good practice to normalize features that use different scales and ranges.

One reason this is important is because the features are multiplied by the model weights. So, the scale of the outputs and the scale of the gradients are affected by the scale of the inputs.

Although a model might converge without feature normalization, normalization makes training much more stable.

In [ ]:
train_dataset.describe().transpose()[['mean','std']]

The features that would benefit from normalization are:

1. age
2. bmi
3. children

The other features are either binary or one-hot encoded (i.e. not continuous and do not need to be normalized, but they are anyway for convenience and simplicity of coding)

We can normalize the features by adding a normalization layer to our model: `tf.keras.layers.Normalization`

In [ ]:
normalizer = layers.Normalization()

Then, fit the state of the preprocessing layer to the data by calling `Normalization.adapt`:

In [ ]:
normalizer.adapt(np.array(train_dataset))

Show the computed means of the features:

In [ ]:
print(normalizer.mean.numpy())

In [ ]:
first = np.array(train_dataset[:1])

print(train_dataset.columns)
with np.printoptions(precision=2, suppress=True):
  print('First example (i.e. 1st row):', first)
  print()
  print('Normalized:', normalizer(first).numpy())

## Test: Linear regression with one variable

Let's see what the relation is between age and healthcare costs

In [ ]:
# Extract a single feature (age) as a np array and normalize it
age = np.array(train_dataset['age'])
age_normalizer = layers.Normalization(input_shape=[1,], axis=None)
age_normalizer.adapt(age)

# Median is around 39, if the normalization was done right
# ages < 39 should be negative, and over positiv
print(age_normalizer(age[:5]).numpy())
print(train_dataset['age'][:5])

Build the Keras Sequential model:

In [ ]:
age_model = tf.keras.Sequential([
    age_normalizer,
    layers.Dense(units=1, use_bias=True)
])

age_model.summary()

This model will predict `'expenses'` from `'age'`.

Run the untrained model on the first 10 'age' values. The output won't be good, but notice that it has the expected shape of `(10, 1)`:

In [ ]:
age

In [ ]:
age_model.predict(age[:10])

Once the model is built, configure the training procedure using the Keras `Model.compile` method. The most important arguments to compile are the `loss` and the `optimizer`, since these define what will be optimized (`mean_absolute_error`) and how (using the `tf.keras.optimizers.Adam`).

In [ ]:
age_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=10.0),
    loss='mean_absolute_error')

In [ ]:
train_labels

Use Keras `Model.fit` to execute the training for 100 epochs:

In [ ]:
# %%time measures how long it takes to train the model
%%time
history = age_model.fit(
    train_dataset['age'],
    train_labels,
    epochs=50,
    # Suppress logging.
    verbose=0,
    # Calculate validation results on 0% of the training data.
    validation_split = 0.2)

In [ ]:
age_model.predict(age[:10])

In [ ]:
hist = pd.DataFrame(history.history)
hist['epoch'] = history.epoch
hist.head(n=10)

In [ ]:
hist.tail(n=5)

In [ ]:
def plot_loss(history, y_min, y_max):
  plt.plot(history.history['loss'], label='loss')
  plt.plot(history.history['val_loss'], label='val_loss')
  plt.ylim([y_min, y_max])
  plt.xlabel('Epoch')
  plt.ylabel('Error [expenses]')
  plt.legend()
  plt.grid(True)

In [ ]:
plot_loss(history, 6000, 13_500)

Collect the results on the dest dataset for later

In [ ]:
test_results = {}

test_results['age_model'] = age_model.evaluate(
    test_dataset['age'],
    test_labels, verbose=0)

Since this is a single variable regression, it's easy to view the model's predictions as a function of the input:

In [ ]:
x = tf.linspace(0.0, 100, 100)
y = age_model.predict(x)

In [ ]:
def plot_age(x, y):
  plt.scatter(train_dataset['age'], train_labels, label='Data')
  plt.plot(x, y, color='k', label='Predictions')
  plt.xlabel('age')
  plt.ylabel('expenses')
  plt.legend()

In [ ]:
plot_age(x, y)

Clearly age alone is a poor predictor of expenses since there seem to be 2 other tranches of data whose expenses are poorly predicted by age alone.

### Linear regression with multiple inputs

We can use an almost identical setup to make predictions based on multiple inputs. This model still does the same $y = mx+b$ except that $m$ is a matrix and $b$ is a vector.

Create a two-step Keras Sequential model again with the first layer being `normalizer` (`tf.keras.layers.Normalization(axis=-1)`) defined earlier and adapted to the whole dataset:

In [ ]:
linear_model = tf.keras.Sequential([
    # Remember: We created the `normalizer` layer earlier for the entire
    # training dataset
    normalizer,
    layers.Dense(units=1, use_bias=True)
])

When we call `Model.predict` on a batch of inputs, it produces `units=1` outputs for each example:

In [ ]:
linear_model.predict(train_dataset[:10])

When we call the model, its weight matrices will be built—check that the `kernel` weights (the $m$ in $y=mx+b$) have a shape of `(9, 1)`:

In [ ]:
linear_model.layers[1].kernel

Configure the model with Keras `Model.compile` and train with `Model.fit` for 100 epochs:

In [ ]:
linear_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=10.0),
    loss='mean_absolute_error')

In [ ]:
%%time
history_multi = linear_model.fit(
    train_dataset,
    train_labels,
    epochs=100,
    # Suppress logging.
    verbose=0,
    # Calculate validation results on 20% of the training data.
    validation_split = 0.2)

In [ ]:
hist_multi = pd.DataFrame(history_multi.history)
hist_multi['epoch'] = history_multi.epoch
hist_multi.head()

In [ ]:
hist_multi.tail()

Using all the inputs in this regression model achieves a much lower training and validation error than the `age_model`, which had one input:

In [ ]:
plot_loss(history_multi, 3_000, 14_000)

Collect the results on the test set for later:

In [ ]:
test_results['linear_model'] = linear_model.evaluate(
    test_dataset, test_labels, verbose=0)

The results are much better, but we're still not cracking the mean average error cut-off of 3,500. Next I will attempt to do the regression with a deep neural network

## Regression with a deep neural network (DNN)

## 1. Regression with a DNN and a single input (age)

In [ ]:
# @param: norm - the normalization layer
def build_and_compile_model(norm):
  model = keras.Sequential([
      norm,
      layers.Dense(64, activation='relu'),
      layers.Dense(64, activation='relu'),
      layers.Dense(1)
  ])

  model.compile(loss='mean_absolute_error',
                optimizer=tf.keras.optimizers.Adam(0.1),
                # The test at the end expects both MAE and MSE to be measured
                metrics=[keras.metrics.mean_absolute_error, keras.metrics.mean_squared_error])
  return model

This model has quite a few more trainable parameters than the linear models:

In [ ]:
dnn_age_model = build_and_compile_model(age_normalizer)

In [ ]:
dnn_age_model.summary()

Train the model with Keras `Model.fit`:

In [ ]:
%%time
history_dnn_age = dnn_age_model.fit(
    train_dataset['age'],
    train_labels,
    validation_split=0.2,
    verbose=0, epochs=50)

In [ ]:
dnn_age_hist = pd.DataFrame(history_dnn_age.history)
dnn_age_hist.head()

In [ ]:
dnn_age_hist.tail()

This model converges faster than the linear single-input `age_model`, and it arrives at a similar result, indicating that the fit is probably as good as it can be for a single variable.

In [ ]:
plot_loss(history_dnn_age, 5_500, 12_000)

If you plot the predictions as a function of `'age'`, you should notice how this model takes advantage of the nonlinearity provided by the hidden layers:

In [ ]:
x = tf.linspace(0.0, 70, 70)
y = dnn_age_model.predict(x)

In [ ]:
plot_age(x, y)

Collect the results on the test set for later:

In [ ]:
test_results['dnn_age_model'] = dnn_age_model.evaluate(
    test_dataset['age'], test_labels,
    verbose=0)

## 2. Regression with a DNN and multiple inputs

Repeat the previous process using all the inputs. The model's performance slightly improves on the validation dataset.

In [ ]:
dnn_model = build_and_compile_model(normalizer)
dnn_model.summary()

In [ ]:
%%time
history_dnn = dnn_model.fit(
    train_dataset,
    train_labels,
    validation_split=0.2,
    verbose=0, epochs=100)

In [ ]:
plot_loss(history_dnn, 1_000, 10_000)

Collect the results on the test set:

In [ ]:
test_results['dnn_model'] = dnn_model.evaluate(test_dataset, test_labels, verbose=0)

## Try to pass the certification test

In [ ]:
# The best model was the multi-input DNN model
model = dnn_model
score = model.evaluate(test_dataset, test_labels, verbose=2)
score

In [ ]:
# Compare the different models
print("|Model name|Score (MAE)|MAE|MSE|")
test_results

The DNN model is good enough to predict a person's healthcare costs to about $1,700 just from knowing the number of children, sex, age, smoking habits and region!

Interestingly the multi-linear regression is also a good enough model to pass the certification test (coming in under the wire)!

In [ ]:
# RUN THIS CELL TO TEST YOUR MODEL. DO NOT MODIFY CONTENTS.
# Test model by checking how well the model generalizes using the test set.
loss, mae, mse = model.evaluate(test_dataset, test_labels, verbose=2)

print("Testing set Mean Abs Error: {:5.2f} expenses".format(mae))

if mae < 3500:
  print("You passed the challenge. Great job!")
else:
  print("The Mean Abs Error must be less than 3500. Keep trying.")

# Plot predictions.
test_predictions = model.predict(test_dataset).flatten()

a = plt.axes(aspect='equal')
plt.scatter(test_labels, test_predictions)
plt.xlabel('True values (expenses)')
plt.ylabel('Predictions (expenses)')
lims = [0, 50000]
plt.xlim(lims)
plt.ylim(lims)
_ = plt.plot(lims,lims)


Looks like there are about 20 outliers that could no be predicted.